Evaluación de **PaddleOCR**, un OCR clásico, e implementación **RTDETR** para detección de vehículos, personas y matrículas en vídeo.

**IMPORTS Y CONFIGURACIÓN GENERAL**

Importaciones únicas para todo el cuaderno (antes estaban repetidas en las dos celdas), clonado del repositorio RT-DETR si no existe, y selección del dispositivo (GPU si está disponible).

In [1]:
import os, sys, subprocess
import csv
import re
import json
import cv2
import numpy as np
import torch
from pathlib import Path
from time import perf_counter
from collections import defaultdict
from PIL import Image
import supervision as sv

from paddleocr import PaddleOCR

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#Cargamos el modelo y ponemos la ruta en memoria
if not os.path.exists("RT-DETR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lyuwenyu/RT-DETR.git"], check=True)


RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")
if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from load_rtdetr import load_rtdetr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**MODELO DE MATRÍCULAS**

Se carga una sola vez el detector de matrículas (RT-DETRv2 tamaño `s`). Antes se cargaba dos veces: como `detector_placas` en la evaluación y como `model_plate` en el vídeo. Es el mismo modelo, así que ahora ambas secciones comparten `model_plate` / `postprocessor_plate`.

In [2]:
PLATE_NAME = {0: "plate"}
PLATE_CLASS = PLATE_NAME.keys()
PLATE_COLOR = {0: (128, 255, 128)}

model_plate, postprocessor_plate = load_rtdetr(family="v2", size="s", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 's' en cuda


**FUNCIONES COMUNES (OCR Y UTILIDADES)**

Funciones que usan tanto la evaluación como el procesamiento de vídeo: gestión de PaddleOCR, normalización de texto, distancia de Levenshtein, recorte de regiones, lectura de matrícula y preprocesado de imagen. Estaban duplicadas; se mantiene la versión de `leer_placa` más completa (con el soporte adicional para la API antigua `.ocr`, que no altera el comportamiento con `.predict`).

In [3]:
# =====================================================
# GESTIÓN Y FUNCIONES OCR (PADDLEOCR)
# =====================================================

ocr_global = None
_ocr_disponible = True

def get_ocr():
    """Devuelve la instancia global de PaddleOCR, inicializándola solo la primera vez."""
    global ocr_global, _ocr_disponible
    if not _ocr_disponible:
        return None
    if ocr_global is None:
        try:
            ocr_global = PaddleOCR(
                lang='es',
                use_textline_orientation=True,       # sustituye a use_angle_cls (deprecado)
                use_doc_orientation_classify=False,  # no necesario para matrículas
                use_doc_unwarping=False,             # evita cargar UVDoc
                device='cpu',
                enable_mkldnn=False,
            )
        except ImportError:
            print("AVISO: paddleocr no está instalado (pip install paddlepaddle paddleocr)")
            _ocr_disponible = False
            return None
    return ocr_global

# Inicializamos el OCR globalmente
ocr_global = get_ocr()

# Normaliza texto a mayúsculas alfanuméricas
def norm(s: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", s.upper())

# Distancia de Levenshtein entre dos cadenas de caracteres
def lev(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

# Recorta una región con margen porcentual o en píxeles ajustada a los límites de la imagen
def recortar(img, box, pad=0.05, margen_px=None):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    if margen_px is not None:
        return img[max(0, int(y1 - margen_px)):min(h, int(y2 + margen_px)),
                   max(0, int(x1 - margen_px)):min(w, int(x2 + margen_px))]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    y2 = min(h, int(y2 + pad * bh))
    return img[y1:y2, x1:x2]

def leer_placa(crop, ocr_inst=None):
    """Devuelve (texto normalizado, confianza media del OCR) usando PaddleOCR."""
    if crop is None or crop.size == 0:
        return "", 0.0

    reader = ocr_inst if ocr_inst is not None else get_ocr()
    if reader is None:
        return "", 0.0

    # Escala mínima garantizada tanto en alto como en ancho
    h_c, w_c = crop.shape[:2]
    esc = max(max(1.0, 64.0 / h_c), max(1.0, 200.0 / w_c))

    g = cv2.resize(crop, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    g = cv2.GaussianBlur(g, (3, 3), 0)
    g = cv2.copyMakeBorder(g, 10, 10, 10, 10, cv2.BORDER_REPLICATE)

    out = []
    if hasattr(reader, "predict"):
        res = reader.predict(g) or []
        for page in res:
            rec_texts = page.get("rec_texts", [])
            rec_scores = page.get("rec_scores", [])
            rec_polys = page.get("rec_polys", [])
            if rec_polys is not None and len(rec_polys) == len(rec_texts):
                for t, s, p in zip(rec_texts, rec_scores, rec_polys):
                    p = np.asarray(p)
                    out.append((t, float(s), float(p[:, 0].mean()), float(p[:, 1].mean())))
            else:
                for t, s in zip(rec_texts, rec_scores):
                    out.append((t, float(s), 0.0, 0.0))
    elif hasattr(reader, "ocr"):
        res = reader.ocr(g, cls=False) or []
        for line in res:
            if isinstance(line, list):
                for item in line:
                    poly, (t, s) = item
                    p = np.asarray(poly)
                    out.append((t, float(s), float(p[:, 0].mean()), float(p[:, 1].mean())))

    out = [r for r in out if norm(r[0])]
    if not out:
        return "", 0.0

    # Ordenar por línea (vertical) y luego de izquierda a derecha
    alto = g.shape[0]
    out.sort(key=lambda r: (round(r[3] / (alto / 2)), r[2]))

    texto = norm("".join(r[0] for r in out))
    confianza = float(np.mean([r[1] for r in out]))
    return texto, confianza

# el preprocesado se separa del detector para hacerlo UNA sola vez por frame
def preprocess(pil_rgb):
    im = pil_rgb.resize((640, 640), Image.BILINEAR)
    return torch.from_numpy(np.array(im)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\vidal\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\vidal\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\vidal\.paddlex\official_models\PP-OCRv6_medium_rec`.


**EVALUACIÓN DEL OCR SOBRE EL DATASET**

Evalúa el detector de matrículas + PaddleOCR sobre imágenes anotadas, comparando contra el texto real (exact match y acierto por carácter), y mide el techo del OCR leyendo directamente el recorte de la anotación.

**Errores corregidos respecto a la versión anterior:**

1. **Tamaño pasado al postprocesador en orden incorrecto.** Se usaba `[[h, w]]`, pero el postprocesador de RT-DETR multiplica las cajas por `orig_target_sizes.repeat(1, 2)`, es decir, espera `(w, h)`. En imágenes no cuadradas las cajas salían mal escaladas, de modo que el IoU contra la anotación caía por debajo de `IOU_MATCH` y las detecciones contaban como falsos positivos geométricos. Es la causa más probable de que la evaluación diera resultados peores de los reales.
2. **Imagen en BGR enviada como RGB.** `cv2.imread` devuelve BGR y se pasaba directamente a `Image.fromarray`; el detector recibía los canales de color intercambiados. En el vídeo sí se convertía a RGB.
3. **Inferencia sin `torch.no_grad()`**: se construía el grafo de gradientes en cada imagen (más memoria y más lento).
4. **`from PIL import Image` ausente** en la celda original (`NameError`; ahora está en los imports comunes).
5. **Guardado de recortes de error**: si el recorte quedaba vacío, `cv2.imwrite` lanzaba excepción; ahora se omite.
6. **Texto GT vacío tras normalizar** (por ejemplo `"???"`) se trata como sin texto en lugar de compararlo como cadena vacía.

El `KeyboardInterrupt` que aparecía en la salida ocurrió dentro de `import torch`: fue una interrupción manual de la celda mientras cargaba las DLL, no un fallo del código.

In [ ]:
# Configuración de rutas y parámetros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
GT_KEYS = ("lpn", "text", "plate", "label", "number", "plate_number")
CONF_METRICAS = 0.35
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("ocr_eval")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "errores-rtdetr").mkdir(exist_ok=True)

detector_placas = model_plate   # mismo modelo que se usa en el vídeo

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

# Carga las cajas y textos anotados desde los archivos JSON
def cargar_gt(img_path, w, h):
    ruta = ANOTACIONES / f"{img_path.stem}.json"
    if not ruta.exists():
        return []
    gts = []
    for obj in json.loads(ruta.read_text()):
        xc, yc, bw, bh = obj["rectangle"]
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        txt = next((norm(str(obj[k])) for k in GT_KEYS if obj.get(k)), None) or None
        gts.append((box, txt))
    return gts


# Muestra el resumen de exact match y acierto medio por carácter
def resumen_metricas(nombre, rows):
    rows = [r for r in rows if r["gt"]]
    if not rows:
        return
    exact = np.mean([r["gt"] == r["ocr"] for r in rows])
    acc_car = np.mean([1 - min(1, lev(r["gt"], r["ocr"]) / max(len(r["gt"]), 1)) for r in rows])
    print(f"{nombre:<34} n={len(rows):<5} exact match={exact:.3f} | acierto por carácter={acc_car:.3f}")

def evaluar_dataset(lista_imagenes, detector=detector_placas):
    """Evalúa OCR en detecciones de RT-DETR y en recortes anotados (techo OCR), midiendo tiempos por detección con perf_counter."""
    filas = []
    n_gt = n_gt_texto = n_detectadas = 0
    tiempos_ocr_det = []
    tiempos_ocr_gt = []
    falsos_positivos_geom = 0
    detecciones_en_gt_sin_texto = 0

    for img_path in lista_imagenes:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = cargar_gt(img_path, w, h)
        n_gt += len(gts)
        n_gt_texto += sum(t is not None for _, t in gts)

        # Inferencia de detección de placas
        # CORRECCIÓN: la imagen se pasa a RGB y el tamaño al postprocesador va como (w, h)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sizes = torch.tensor([[w, h]], device=DEVICE)

        with torch.no_grad():
            labels, boxes, scores = postprocessor_plate(detector(preprocess(Image.fromarray(rgb))), sizes)

        mask = scores >= CONF_METRICAS


        preds = [(b.tolist(), float(s)) for b, s in zip(boxes[mask], scores[mask])]
        n_detectadas += len(preds)

        # OCR a partir de las detecciones de RT-DETR
        for box, det_conf in preds:
            t_ini_det = perf_counter()
            rec = recortar(img, box, pad=PAD)
            texto, ocr_conf = leer_placa(rec)
            tiempos_ocr_det.append(perf_counter() - t_ini_det)

            mejor_iou, gt_txt = 0.0, None
            for gbox, gtxt in gts:
                v = iou(box, gbox)
                if v > mejor_iou:
                    mejor_iou, gt_txt = v, gtxt

            # Análisis de emparejamiento con el Ground Truth
            if mejor_iou < IOU_MATCH:
                falsos_positivos_geom += 1
                continue
            elif gt_txt is None or gt_txt == "NONE":
                detecciones_en_gt_sin_texto += 1
                continue

            filas.append(dict(imagen=img_path.name, origen="deteccion", gt=gt_txt, ocr=texto,
                              det_conf=round(det_conf, 3), ocr_conf=round(ocr_conf, 3),
                              iou=round(mejor_iou, 3)))
            if gt_txt != texto and rec.size > 0:
                cv2.imwrite(str(SALIDA / "errores-rtdetr" / f"{len(filas):05d}_gt-{gt_txt}_ocr-{texto}.jpg"), rec)

        # 2) OCR a partir del recorte de la anotación = techo del OCR
        for gbox, gtxt in gts:
            if gtxt is None or gtxt == "NONE":
                continue
            t_ini_gt = perf_counter()
            rec_gt = recortar(img, gbox, pad=PAD)
            texto, ocr_conf = leer_placa(rec_gt)
            tiempos_ocr_gt.append(perf_counter() - t_ini_gt)

            filas.append(dict(imagen=img_path.name, origen="gt_recorte", gt=gtxt, ocr=texto,
                              det_conf=None, ocr_conf=round(ocr_conf, 3), iou=1.0))

    # Guardar resultados en CSV
    csv_path = SALIDA / "rt-detr_resultados_ocr.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()) if filas else ["vacio"], delimiter=";")
        wr.writeheader()
        wr.writerows(filas)

    # Métricas de tiempo medio de ejecución por detección (extra)
    tiempo_medio_ocr_det = (np.mean(tiempos_ocr_det) * 1000) if tiempos_ocr_det else 0.0
    tiempo_medio_ocr_gt = (np.mean(tiempos_ocr_gt) * 1000) if tiempos_ocr_gt else 0.0

    print(f"\nImágenes evaluadas: {len(lista_imagenes)} | placas anotadas: {n_gt} (con texto: {n_gt_texto}) | detecciones RT-DETR: {n_detectadas}")
    print(f"--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---")
    print(f"OCR a partir de detección RT-DETR:     {tiempo_medio_ocr_det:6.2f} ms / detección")
    print(f"OCR a partir de recorte GT (techo): {tiempo_medio_ocr_gt:6.2f} ms / recorte anotado")

    if n_gt_texto == 0:
        print(f"\nLas anotaciones no contienen texto de matrícula (claves probadas: {GT_KEYS}).")
    else:
        print(f"\n--- RESUMEN DE EXACTITUD OCR ---")
        resumen_metricas("OCR sobre detecciones de RT-DETR", [r for r in filas if r["origen"] == "deteccion"])
        resumen_metricas("OCR sobre recorte anotado (techo)", [r for r in filas if r["origen"] == "gt_recorte"])
        emp = sum(1 for r in filas if r["origen"] == "deteccion" and r["gt"])
        print(f"Placas evaluadas encontradas por RT-DETR: {emp}/{n_gt_texto}")
        print(f"Falsos positivos geométricos (IoU < {IOU_MATCH}): {falsos_positivos_geom}")
        print(f"Detecciones en placas anotadas sin texto (None): {detecciones_en_gt_sin_texto}")

    print(f"\nCSV: {csv_path}\nRecortes con error: {SALIDA / 'errores-rtdetr'}")
    return filas

# Ejecución de la evaluación de métricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")


Imágenes evaluadas: 200 | placas anotadas: 407 (con texto: 407) | detecciones RT-DETR: 567
--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---
OCR a partir de detección RT-DETR:     421.05 ms / detección
OCR a partir de recorte GT (techo): 636.97 ms / recorte anotado

--- RESUMEN DE EXACTITUD OCR ---
OCR sobre detecciones de RT-DETR   n=188   exact match=0.702 | acierto por carácter=0.881
OCR sobre recorte anotado (techo)  n=210   exact match=0.648 | acierto por carácter=0.791
Placas evaluadas encontradas por RT-DETR: 188/407
Falsos positivos geométricos (IoU < 0.5): 278
Detecciones en placas anotadas sin texto (None): 101

CSV: ocr_eval\rt-detr_resultados_ocr.csv
Recortes con error: ocr_eval\errores-rtdetr


**PARÁMETROS DEL VÍDEO Y MODELO DE VEHÍCULOS**

Clases, colores y parámetros de detección, tracking y OCR sobre vídeo. Se carga aquí el detector de vehículos y personas (RT-DETRv2 tamaño `l`), que solo hace falta para esta sección; el de matrículas ya está cargado arriba.

In [ ]:
CLASS_NAMES = {0: 'person', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}  # 0..79
BASE_CLASS = CLASS_NAMES.keys()
BASE_COLORS = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255), 5: (0, 255, 255), 7: (128, 128, 255)}

# Las matriculas se buscan dentro del recorte de cada vehiculo y heredan su tracker_id
MAX_CROPS = 8          # maximo de vehiculos por frame donde se buscan matriculas
MIN_CROP_SIDE = 48     # px: vehiculos mas pequenos que esto se ignoran
CROP_MARGIN = 0.05     # margen extra alrededor del vehiculo
PLATE_CONF = 0.3      # confianza minima del detector de matriculas
HOLD_FRAMES = 5        # frames que se mantiene la caja de matricula si no se detecta

# Parámetros visuales y de OCR de matrículas
COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

# Patrones sintácticos de matrículas europeas
CONS = "BCDFGHJKLMNPRSTVWXYZ"
PATRONES_MATRICULA = [re.compile(p) for p in (
    rf"\d{{4}}[{CONS}]{{3}}",              # España actual
    r"[A-Z]{1,2}\d{4}[A-Z]{1,2}",          # España antigua
    r"[A-Z]{2}\d{3}[A-Z]{2}",              # Francia / Italia
    r"[A-Z]{2}\d{2}[A-Z]{3}",              # Reino Unido
    r"[A-Z]{1,3}[A-Z]{1,2}\d{1,4}[EH]?",   # Alemania
    r"\d{2}[A-Z]{2}\d{2}",                 # Portugal
    r"[A-Z]{2}\d{2}[A-Z]{2}",              # Portugal
    r"\d{2}\d{2}[A-Z]{2}",                 # Portugal
    r"[1-9][A-Z]{3}\d{3}",                 # Bélgica
    r"\d{2}[A-Z]{3}\d",                    # Países Bajos
    r"\d[A-Z]{3}\d{2}",
    r"[A-Z]{2}\d{3}[A-Z]",
    r"\d{3}[A-Z]{2}\d",
    r"[A-Z]\d{3}[A-Z]{2}",
    r"[A-Z]{3}\d{2}[A-Z]",
    r"[A-Z]{3}\d{3}",                      # Polonia / otros
    r"[A-Z]{2}\d{5}",
)]

# Valida si el texto coincide con alguna sintaxis europea conocida
def matricula_valida(texto):
    return any(p.fullmatch(texto) for p in PATRONES_MATRICULA)

model, postprocessor = load_rtdetr(family="v2", size="l", device=DEVICE)

**TRACKING Y DETECCIÓN RT-DETR**

Funciones del tracker ByteTrack, letterbox, ejecución del detector y búsqueda de matrículas dentro del recorte de cada vehículo.

In [ ]:
def make_tracker(buffer_seconds, min_conf=0.7):
    return sv.ByteTrack(
        track_activation_threshold=min_conf,                   # score mínimo para iniciar un track
        lost_track_buffer=int(fps * buffer_seconds),      # frames que aguanta un track sin detección
        minimum_matching_threshold=0.7,
        frame_rate=int(fps),
        minimum_consecutive_frames=4,
    )

# Letterbox, mantiene la proporcion del recorte y rellena con gris
def letterbox(pil_rgb, size=640):
    w, h = pil_rgb.size
    r = size / max(w, h)                                  # escala unica: no deforma
    nw, nh = int(round(w * r)), int(round(h * r))
    canvas = Image.new("RGB", (size, size), (114, 114, 114))
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas.paste(pil_rgb.resize((nw, nh), Image.BILINEAR), (px, py))
    x = torch.from_numpy(np.array(canvas)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)
    return x, r, px, py

@torch.no_grad()
def run_detector(x, mdl, post, conf, valid_classes, sizes):
    # x: tensor (B,3,640,640) ya preprocesado | sizes: tensor (B,2) con (w,h) originales
    labels, boxes, scores = post(mdl(x), sizes)
    labels, boxes, scores = labels.cpu().numpy(), boxes.cpu().numpy(), scores.cpu().numpy()
    results = []
    for l, b, s in zip(labels, boxes, scores):
        keep = (s >= conf) & np.isin(l, np.array(valid_classes))
        results.append((b[keep], s[keep], l[keep]))
    return results   # lista con una tupla (boxes, scores, labels) por cada imagen del batch

# Devuelve {tracker_id_vehiculo: (score, caja_en_frame)} con la mejor matricula de cada vehiculo
def detect_plates_in_crops(rgb, veh_xyxy, veh_tids, conf):
    H, W = rgb.shape[:2]
    if len(veh_xyxy) == 0:
        return {}

    areas = (veh_xyxy[:, 2] - veh_xyxy[:, 0]) * (veh_xyxy[:, 3] - veh_xyxy[:, 1])
    order = np.argsort(-areas)[:MAX_CROPS]          # los vehiculos mas grandes primero

    xs, meta = [], []
    for i in order:
        x1, y1, x2, y2 = veh_xyxy[i]
        bw, bh = x2 - x1, y2 - y1
        if bw < MIN_CROP_SIDE or bh < MIN_CROP_SIDE:
            continue
        mx, my = bw * CROP_MARGIN, bh * CROP_MARGIN
        cx1, cy1 = int(max(0, x1 - mx)), int(max(0, y1 - my))
        cx2, cy2 = int(min(W, x2 + mx)), int(min(H, y2 + my))
        if cx2 - cx1 < 2 or cy2 - cy1 < 2:
            continue
        x, r, px, py = letterbox(Image.fromarray(rgb[cy1:cy2, cx1:cx2]))
        xs.append(x)
        meta.append((int(veh_tids[i]), cx1, cy1, r, px, py, (x1, y1, x2, y2)))

    if not xs:
        return {}

    sizes = torch.tensor([[640, 640]] * len(xs), device=DEVICE)
    results = run_detector(torch.cat(xs), model_plate, postprocessor_plate,
                           conf, list(PLATE_CLASS), sizes)

    best = {}   # tid_vehiculo -> (score, caja)
    for (b, s, l), (tid, ox, oy, r, px, py, vb) in zip(results, meta):
        for bb, ss in zip(b, s):
            # deshacer letterbox y volver a coordenadas del frame
            box = np.array([(bb[0] - px) / r + ox, (bb[1] - py) / r + oy,
                            (bb[2] - px) / r + ox, (bb[3] - py) / r + oy])
            cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
            if not (vb[0] <= cx <= vb[2] and vb[1] <= cy <= vb[3]):
                continue   # matricula del margen que pertenece a otro coche
            if tid not in best or ss > best[tid][0]:
                best[tid] = (float(ss), box)
    return best

**BUCLE PRINCIPAL DE PROCESAMIENTO DE VÍDEO**

Detecta personas y vehículos, les asigna identificadores con ByteTrack, busca la matrícula de cada vehículo, lee el texto con PaddleOCR y escribe el vídeo anotado y un CSV con todos los resultados.

In [ ]:
path_test = 'C0142.MP4'
output_dir = os.path.abspath('RT-DETR_PaddleOCR_result')
os.makedirs(output_dir, exist_ok=True)
output_path =  os.path.join(output_dir, 'resultado.mp4')
output_csv = os.path.join(output_dir, 'resultado.csv')
vid = cv2.VideoCapture(path_test)

w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = vid.get(cv2.CAP_PROP_FPS) or 30

# escritor de video a disco (sustituye a la ventana)
writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# csv con separador punto y coma
csv_file = open(output_csv, 'w', newline='', encoding='utf-8-sig')
csv_writer = csv.writer(csv_file, delimiter=';')
csv_writer.writerow(['fotograma', 'tipo_objeto', 'confianza', 'identificador_tracking',
                     'x1', 'y1', 'x2', 'y2',
                     'matricula_en_su_caso', 'confianza_matricula', 'mx1', 'my1', 'mx2', 'my2',
                     'texto_matricula'])

tracker_person = make_tracker(4)    # peatones: aguantan 4 s tapados
tracker_vehicle = make_tracker(1.5) # 3s
# tracker_plate no existe ya que la matricula hereda el id del vehiculo

size_full = torch.tensor([[w, h]], device=DEVICE)

ids_propios = {c: {} for c in CLASS_NAMES}
class_scores = defaultdict(lambda: defaultdict(float))  # key -> {clase: score acumulado}
track_class = {}                                         # key -> clase votada actual
next_num = {c: 1 for c in CLASS_NAMES}           # siguiente número visible por clase

plates_seen = set()      # vehiculos que han tenido matricula alguna vez
last_plate = {}          # tid_vehiculo -> (score, caja, frame_idx)

# Estructuras de tracking y lectura OCR
ultimo_ocr = {}                 # tid_vehiculo -> frame_idx del último intento OCR
intentos_ocr = defaultdict(int) # tid_vehiculo -> número de intentos OCR realizados
lecturas = {}                   # tid_vehiculo -> (score_total, texto_matricula)

frame_idx = 0

while vid.isOpened():
    ret, image = vid.read()

    if not ret:
      break

    # conversion y resize UNA sola vez para el detector de vehiculos
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    x_full = preprocess(Image.fromarray(rgb))

    boxes, scores, labels = run_detector(x_full, model, postprocessor, 0.5, list(BASE_CLASS), size_full)[0]    #NO TOCAR CONF(0.65)

    detections = sv.Detections(
        xyxy=boxes.astype(np.float32),
        confidence=scores.astype(np.float32),
        class_id=labels.astype(int),
    )

    det_person = detections[detections.class_id == list(BASE_CLASS)[0]]
    det_veh = detections[np.isin(detections.class_id,  list(BASE_CLASS)[1:])]
    det_veh = det_veh.with_nms(threshold=0.55, class_agnostic=False)  # duplicados de la misma clase
    det_veh = det_veh.with_nms(threshold=0.65, class_agnostic=True)    # (0.35) mismo objeto con dos clases

    det_person = tracker_person.update_with_detections(det_person)
    det_veh = tracker_vehicle.update_with_detections(det_veh)

    # matriculas: se buscan en el recorte de cada vehiculo trackeado y se asocian a su tracker_id
    plate_by_tid = detect_plates_in_crops(rgb, det_veh.xyxy, det_veh.tracker_id, PLATE_CONF)
    for tid, (sc, bx) in plate_by_tid.items():
        last_plate[tid] = (sc, bx, frame_idx)

        # Inferencia OCR sobre la placa detectada (control de frecuencia y número de intentos)
        ancho_placa = bx[2] - bx[0]
        if (ancho_placa >= MIN_ANCHO_OCR
                and intentos_ocr[tid] < OCR_MAX_INTENTOS
                and lecturas.get(tid, (0.0, ""))[0] < OCR_SCORE_OK
                and (tid not in ultimo_ocr or frame_idx - ultimo_ocr[tid] >= OCR_CADA)):
            ultimo_ocr[tid] = frame_idx
            intentos_ocr[tid] += 1
            rec_placa = recortar(image, (int(bx[0]), int(bx[1]), int(bx[2]), int(bx[3])), margen_px=4)
            texto, oconf = leer_placa(rec_placa)
            if len(texto) >= MIN_CHARS_PLACA and matricula_valida(texto):
                score_ocr = sc * oconf
                if tid not in lecturas or score_ocr > lecturas[tid][0]:
                    lecturas[tid] = (score_ocr, texto)

    detections = sv.Detections.merge([det_person, det_veh])

    frame_rows = []   # datos del frame para el csv

    for box, score, label, tid in zip(detections.xyxy, detections.confidence, detections.class_id, detections.tracker_id):
        # Los dos trackers generan IDs desde 1, así que la clave incluye el grupo
        grupo = 'person' if label == 0 else 'veh'
        key = (grupo, tid)

        # Voto: suma el score de la clase detectada en este frame
        class_scores[key][label] += score
        cls = max(class_scores[key], key=class_scores[key].get)

        # Si la clase votada cambió, mueve el track a la nueva clase (para no inflar el conteo)
        if track_class.get(key) != cls:
            old = track_class.get(key)
            if old is not None:
                ids_propios[old].pop(key, None)
            ids_propios[cls][key] = next_num[cls]
            next_num[cls] += 1
            track_class[key] = cls

        x1, y1, x2, y2 = box.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), BASE_COLORS[cls], 2)
        cv2.putText(image, f" id: {ids_propios[cls][key]} {CLASS_NAMES[cls]}: {score:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, BASE_COLORS[cls], 2)

        frame_rows.append({
            'grupo': grupo, 'cls': cls, 'score': float(score), 'tid': tid,
            'id': ids_propios[cls][key], 'box': (int(x1), int(y1), int(x2), int(y2)),
            'plate': None,
        })

    # escritura del csv (personas y vehiculos; la matricula va en la fila de su vehiculo)
    for r in frame_rows:
        if r['grupo'] == 'veh' and r['tid'] in last_plate:
            sc, bx, f = last_plate[r['tid']]
            if frame_idx - f <= HOLD_FRAMES:
                pb = tuple(int(v) for v in bx)
                txt = lecturas[r['tid']][1] if r['tid'] in lecturas else ""
                r['plate'] = {'id': r['id'], 'score': sc, 'box': pb, 'texto': txt}
                plates_seen.add(('veh', r['tid']))
                # Renderizado de matrícula idéntico a YOLO (pone '?' si no hay lectura válida)
                px1, py1, px2, py2 = pb
                cv2.rectangle(image, (px1, py1), (px2, py2), COLOR_PLACA, 2)
                txt_display = txt if txt else "?"
                fuente, esc, grosor = cv2.FONT_HERSHEY_SIMPLEX, ESCALA_TEXTO_PLACA, 1
                (tw, th), base = cv2.getTextSize(txt_display, fuente, esc, grosor)
                bx1 = max(0, px1)
                by2 = max(th + base + 6, py1 - 2)
                by1 = by2 - th - base - 6
                cv2.rectangle(image, (bx1, by1), (bx1 + tw + 8, by2), COLOR_PLACA, -1)
                cv2.putText(image, txt_display, (bx1 + 4, by2 - base - 3),
                            fuente, esc, (0, 0, 0), grosor)

        p = r['plate']
        plate_cols = ([p['id'], f"{p['score']:.4f}", *p['box'], p['texto']] if p
                      else ['', '', '', '', '', '', ''])
        csv_writer.writerow([frame_idx, CLASS_NAMES[r['cls']], f"{r['score']:.4f}", r['id'],
                             *r['box'], *plate_cols])

    # Barra inferior con los totales
    cv2.rectangle(image, (0, h - 40), (w, h), (0, 0, 0), -1)
    texto = "  |  ".join(f"{CLASS_NAMES[c]}: {len(ids_propios[c])}" for c in CLASS_NAMES)
    texto += f"  |  plate: {len(plates_seen)}"
    texto += f"  |  ocr: {len(lecturas)}"
    cv2.putText(image, texto, (10, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    writer.write(image)
    frame_idx += 1

vid.release()
writer.release()
csv_file.close()

print(f"Video guardado en {output_path} | CSV guardado en {output_csv}")
print(f"Matrículas detectadas: {len(plates_seen)} | Matrículas leídas con OCR: {len(lecturas)}")